In [1]:
import io
import os
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path

warnings.filterwarnings("ignore")


# ============================================================
# 1. PLOT GENERATION
# ============================================================

def generate_box_plot(all_data, save_filename):
    fig, ax = plt.subplots(figsize=(11, 8))

    libraries = sorted(all_data.keys())
    data_to_plot = [all_data[lib]["scores"] for lib in libraries]

    ax.boxplot(
        data_to_plot,
        labels=libraries,
        patch_artist=True,
        showmeans=True,
        meanprops=dict(
            marker="x",
            markerfacecolor="black",
            markeredgecolor="black",
            markersize=9,
            markeredgewidth=2,
            zorder=4,
        ),
        boxprops=dict(
            facecolor="#dbe9f6",
            color="darkblue",
            linewidth=1.5,
        ),
        medianprops=dict(color="crimson", linewidth=2.5),
        whiskerprops=dict(color="darkblue", linewidth=1.5),
        capprops=dict(color="darkblue", linewidth=1.5),
        flierprops=dict(
            marker="o",
            markerfacecolor="gray",
            alpha=0.5,
            markersize=5,
            markeredgecolor="none",
        ),
    )

    ax.set_xlabel(
        "Screened Library Name",
        fontsize=12, weight="bold", labelpad=10
    )
    ax.set_ylabel(
        "GOLD Fitness Score",
        fontsize=12, weight="bold", labelpad=10
    )
    ax.grid(True, linestyle=":", alpha=0.6, zorder=1)

    plt.xticks(rotation=30, ha="right", fontsize=10)
    plt.tight_layout()
    plt.savefig(save_filename, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)


# ============================================================
# 2. PARSE GOLD FILES
# ============================================================

def parse_gold_file(content):
    columns = [
        "Score", "S(PLP)", "S(hbond)", "S(cho)", "S(metal)",
        "DE(clash)", "DE(tors)", "intcor", "RMS(heavy)", "time",
        "File_name", "Ligand_name",
    ]

    if isinstance(content, memoryview):
        content = content.tobytes()

    if isinstance(content, bytes):
        content = content.decode("utf-8-sig", errors="replace")

    data_lines = [
        line for line in content.splitlines()
        if line.strip() and not line.strip().startswith("#")
    ]

    if not data_lines:
        raise ValueError("The file contains no data rows.")

    df = pd.read_csv(
        io.StringIO("\n".join(data_lines)),
        sep=r"\s+",
        names=columns,
        header=None,
    )

    scores = (
        pd.to_numeric(df["Score"], errors="coerce")
        .dropna()
        .tolist()
    )

    if not scores:
        raise ValueError("No valid GOLD fitness scores were found.")

    return scores


# ============================================================
# 3. INTERACTIVE DASHBOARD
# ============================================================

class BoxPlotApp:

    def __init__(self):
        self.uploaded_files = []
        self.file_index = 0
        self.all_data = {}

        self.output = widgets.Output()

        self.uploader = widgets.FileUpload(
            description="Choose GOLD Files",
            accept=".lst,.txt,*",
            multiple=True,
            layout=widgets.Layout(width="300px"),
        )

        self.start_button = widgets.Button(
            description="Process Uploaded Files",
            button_style="primary",
            icon="play",
        )

        self.start_button.on_click(self.start_processing)

        display(widgets.HTML(
            "<h3>GOLD Docking Score Comparison</h3>"
            "<p>Select one or more GOLD result files from your computer, "
            "then click Process Uploaded Files.</p>"
        ))
        display(self.uploader, self.start_button, self.output)

    # --------------------------------------------------------
    # Normalize FileUpload values across ipywidgets versions
    # --------------------------------------------------------

    def get_uploaded_files(self):
        value = self.uploader.value
        files = []

        # ipywidgets 8: tuple/list of file dictionaries
        if isinstance(value, (tuple, list)):
            for item in value:
                files.append({
                    "name": item["name"],
                    "content": item["content"],
                })

        # ipywidgets 7: dictionary keyed by filename
        elif isinstance(value, dict):
            for name, item in value.items():
                files.append({
                    "name": name,
                    "content": item["content"],
                })

        return files

    # --------------------------------------------------------
    # Start processing uploaded files
    # --------------------------------------------------------

    def start_processing(self, button):
        with self.output:
            clear_output(wait=True)

            self.uploaded_files = self.get_uploaded_files()
            self.file_index = 0
            self.all_data = {}

            if not self.uploaded_files:
                print("Please choose at least one file first.")
                return

            print(
                f"{len(self.uploaded_files)} file(s) selected."
            )

            self.show_next_file()

    # --------------------------------------------------------
    # Configure one library at a time
    # --------------------------------------------------------

    def show_next_file(self):
        with self.output:
            clear_output(wait=True)

            if self.file_index >= len(self.uploaded_files):
                self.finalize()
                return

            item = self.uploaded_files[self.file_index]
            filename = item["name"]
            default_name = os.path.splitext(filename)[0]

            print(
                f"Configure library {self.file_index + 1} "
                f"of {len(self.uploaded_files)}"
            )
            print(f"File: {filename}")

            self.lib_input = widgets.Text(
                value=default_name,
                description="Library:",
                layout=widgets.Layout(width="70%"),
            )

            self.target_input = widgets.Text(
                value="Target Protein",
                description="Protein:",
                layout=widgets.Layout(width="70%"),
            )

            self.save_button = widgets.Button(
                description=(
                    "Save & Next File"
                    if self.file_index < len(self.uploaded_files) - 1
                    else "Save & Generate Plot"
                ),
                button_style="info",
            )

            self.save_button.on_click(self.save_library)

            display(self.lib_input)
            display(self.target_input)
            display(self.save_button)

    # --------------------------------------------------------
    # Save current library details and scores
    # --------------------------------------------------------

    def save_library(self, button):
        with self.output:
            item = self.uploaded_files[self.file_index]

            lib_name = self.lib_input.value.strip()
            target_name = self.target_input.value.strip()

            if not lib_name:
                lib_name = os.path.splitext(item["name"])[0]

            if not target_name:
                target_name = "Unknown Protein"

            try:
                scores = parse_gold_file(item["content"])

                self.all_data[lib_name] = {
                    "scores": scores,
                    "target": target_name,
                }

                print(
                    f"Loaded {len(scores)} scores for {lib_name}."
                )

            except Exception as e:
                print(f"Error processing {item['name']}: {e}")

            self.file_index += 1
            self.show_next_file()

    # --------------------------------------------------------
    # Generate plot and editable caption
    # --------------------------------------------------------

    def finalize(self):
        with self.output:
            clear_output(wait=True)

            if not self.all_data:
                print("No files were successfully processed.")
                return

            self.plot_path = Path.cwd() / "docking_scores_boxplot.png"
            self.caption_path = Path.cwd() / "caption_boxplot.txt"

            generate_box_plot(
                self.all_data,
                str(self.plot_path),
            )

            targets = sorted({
                item["target"] for item in self.all_data.values()
            })
            libraries = sorted(self.all_data.keys())

            caption = (
                "Fig 1. Box and whisker plot comparing the distribution "
                "of overall GOLD fitness scores for the evaluated small "
                f"molecule libraries ({', '.join(libraries)}) screened "
                f"against the protein target(s) ({', '.join(targets)}). "
                "The central horizontal line denotes the median score, "
                "while the 'x' markers represent the calculated sample "
                "means. The box margins define the interquartile range "
                "(IQR), and individual points represent outliers beyond "
                "the distribution whiskers."
            )

            print("\nEdit the figure caption below:")
            self.caption_input = widgets.Textarea(
                value=caption,
                description="Caption:",
                layout=widgets.Layout(width="98%", height="150px"),
            )

            self.finish_button = widgets.Button(
                description="Save Caption and Finish",
                button_style="success",
                icon="save",
            )

            self.finish_button.on_click(self.save_results)

            display(self.caption_input, self.finish_button)

    # --------------------------------------------------------
    # Save final results
    # --------------------------------------------------------

    def save_results(self, button):
        with self.output:
            with open(
                self.caption_path, "w", encoding="utf-8"
            ) as f:
                f.write(self.caption_input.value)

            print("\nProcessing complete!")
            print(f"Plot saved: {self.plot_path}")
            print(f"Caption saved: {self.caption_path}")
            print(
                "\nDownload both files using the JupyterHub "
                "file browser."
            )

            self.finish_button.disabled = True
            self.finish_button.description = "Results Saved"

# ============================================================
# 4. LAUNCH APP
# ============================================================

app = BoxPlotApp()


HTML(value='<h3>GOLD Docking Score Comparison</h3><p>Select one or more GOLD result files from your computer, …

FileUpload(value=(), accept='.lst,.txt,*', description='Choose GOLD Files', layout=Layout(width='300px'), mult…

Button(button_style='primary', description='Process Uploaded Files', icon='play', style=ButtonStyle())

Output()